# Baseline leaderboard

Scans every run (end-to-end + autoencoder classifier, notebook- or sweep-produced), ranks by **balanced accuracy**, and writes `baseline/leaderboard.json`. Old autoencoder runs predate balanced accuracy and are flagged `needs_rerun`.

Balanced accuracy is only directly comparable within the same `num_classes` — filter on that column before comparing.

In [ ]:
import sys
from pathlib import Path

FYP_ROOT = Path('/path/to/BrainWear_Kareem/FYP')
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

from baseline.sweeps import aggregate
import pandas as pd

In [ ]:
leaderboard = aggregate.build_leaderboard(write=True)
print(f"Best: {leaderboard['best']}")

df = pd.DataFrame(leaderboard['ranked'])
cols = ['pipeline', 'dataset', 'model', 'num_classes', 'balanced_accuracy', 'f1', 'val_loss', 'run_name', 'needs_rerun']
df[[c for c in cols if c in df.columns]]

In [ ]:
# Example: compare only 3-class runs, sorted by f1
pd.set_option("display.max_colwidth", None)
df[df['num_classes'] == 3].sort_values('f1', ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Excludes heavily inbalanced classes

# Best by F1 for each (pipeline, dataset) combination
valid = df[~df['needs_rerun'] & df['f1'].notna()].copy()
best_per_combo = (
    valid.sort_values('f1', ascending=False)
         .groupby(['pipeline', 'dataset'], sort=False)
         .first()
         .reset_index()
    [['pipeline', 'dataset', 'model', 'f1', 'balanced_accuracy', 'accuracy', 'run_name']]
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

display(
    best_per_combo
    .style
    .format({'f1': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'accuracy': '{:.3f}'})
    .bar(subset=['f1'], color='steelblue', vmin=0, vmax=1)
    .set_caption('Best run per pipeline x dataset (ranked by macro F1)')
)

labels = [f"{r.pipeline}\n{r.dataset}" for _, r in best_per_combo.iterrows()]
x = np.arange(len(labels))
width = 0.26

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.8), 5))
ax.bar(x - width, best_per_combo['f1'],                width, label='Macro F1',      color='steelblue')
ax.bar(x,          best_per_combo['balanced_accuracy'], width, label='Balanced Acc',  color='coral')
ax.bar(x + width, best_per_combo['accuracy'],           width, label='Accuracy',      color='mediumseagreen')

ax.axhline(1/3, color='grey', linestyle='--', linewidth=0.8, label='Random baseline (3-class)')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title('Best run per pipeline x dataset')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Filter settings ────────────────────────────────────────────────────────────
# Restrict brainwear* runs to one EORTC scale before picking the best per combo.
# Set to None to keep all scores (same behaviour as the cell above).
BRAINWEAR_SCORE_FILTER = "QL2"  
# ───────────────────────────────────────────────────────────────────────────────

valid = df[~df['needs_rerun'] & df['f1'].notna()].copy()

# score_name is emitted by aggregate.py; derive it from disk if df predates that change
if 'score_name' not in valid.columns:
    import json as _json
    def _read_score_name(path):
        for fname, key_path in [('args.json', ['score_name']),
                                 ('cv_run_details.json', ['arguments', 'score_name'])]:
            p = Path(path) / fname
            if p.exists():
                d = _json.loads(p.read_text())
                v = d
                for k in key_path:
                    v = v.get(k) if isinstance(v, dict) else None
                if v:
                    return v
        return None
    valid['score_name'] = valid['path'].apply(_read_score_name)

is_brainwear = valid['dataset'].str.startswith('brainwear')
if BRAINWEAR_SCORE_FILTER is not None:
    valid = valid[~is_brainwear | (valid['score_name'] == BRAINWEAR_SCORE_FILTER)]

best = (
    valid.sort_values('f1', ascending=False)
         .groupby(['pipeline', 'dataset'], sort=False)
         .first()
         .reset_index()
    [['pipeline', 'dataset', 'model', 'score_name', 'f1', 'balanced_accuracy', 'accuracy', 'run_name']]
    .sort_values('f1', ascending=False)
    .reset_index(drop=True)
)

title = (
    f"Best per pipeline × dataset — brainwear* filtered to score='{BRAINWEAR_SCORE_FILTER}'"
    if BRAINWEAR_SCORE_FILTER
    else "Best per pipeline × dataset (all scores)"
)

display(
    best.style
        .format({'f1': '{:.3f}', 'balanced_accuracy': '{:.3f}', 'accuracy': '{:.3f}'})
        .bar(subset=['f1'], color='steelblue', vmin=0, vmax=1)
        .set_caption(title)
)

labels = [f"{r.pipeline}\n{r.dataset}" for _, r in best.iterrows()]
x = np.arange(len(labels))
width = 0.26

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.8), 5))
ax.bar(x - width, best['f1'],                width, label='Macro F1',     color='steelblue')
ax.bar(x,          best['balanced_accuracy'], width, label='Balanced Acc', color='coral')
ax.bar(x + width,  best['accuracy'],          width, label='Accuracy',     color='mediumseagreen')

ax.axhline(1/3, color='grey', linestyle='--', linewidth=0.8, label='Random baseline (3-class)')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_title(title)
ax.legend()
fig.tight_layout()
plt.show()
